## 1. Imports

In [1]:
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix

I0000 00:00:1778046620.346566   59819 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778046620.386106   59819 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778046621.503808   59819 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## 2. Creating the test data generator

In [2]:
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    '../data/Testing',                # or '../data/Testing' if notebook is inside notebooks/
    target_size=(128,128),         # must match the size used during training
    batch_size=32,
    class_mode='categorical',
    shuffle=False                  # very important – keeps order for evaluation
)

Found 1600 images belonging to 4 classes.


## 3. Load the trained model

In [3]:
model = load_model('models/scratch_model.keras')
model.summary()
print("Model loaded.")


I0000 00:00:1778046624.290148   59819 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2143 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     6,422,784 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │         1,028 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,551,182 (74.58 MB)

 Trainable params: 6,517,060 (24.86 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 13,034,122 (49.72 MB)

Model loaded.


## 4. Evaluating overall test accuracy and loss 

In [4]:
test_loss, test_acc = model.evaluate(test_generator, verbose=1)
print(f"\nTest accuracy: {test_acc:.4f}")
print(f"Test loss: {test_loss:.4f}")

I0000 00:00:1778046625.696871   59819 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1778046625.935709   59984 service.cc:153] XLA service 0x75ebcc0402c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778046625.935731   59984 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 3050 Laptop GPU, Compute Capability 8.6 (Driver: 13.0.0; Runtime: 12.8.0; Toolkit: 12.5.0; DNN: 9.10.2)
I0000 00:00:1778046625.968001   59984 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1778046626.029583   59984 cuda_dnn.cc:461] Loaded cuDNN version 91002


 5/50 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.7613 - loss: 4.1207

I0000 00:00:1778046628.233611   59984 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


50/50 ━━━━━━━━━━━━━━━━━━━━ 5s 40ms/step - accuracy: 0.9112 - loss: 1.2190

Test accuracy: 0.9112
Test loss: 1.2190


## 5. Get metrics and confusion matrix (per each class)

In [5]:
# Get predictions
y_pred_probs = model.predict(test_generator)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_generator.classes
class_names = list(test_generator.class_indices.keys())

# Classification report
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names))

# Confusion matrix
print("\nConfusion matrix:")
print(confusion_matrix(y_true, y_pred))

50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step

Classification Report:
              precision    recall  f1-score   support

      glioma       0.97      0.80      0.88       400
  meningioma       0.95      0.85      0.90       400
     notumor       0.80      1.00      0.89       400
   pituitary       0.97      0.99      0.98       400

    accuracy                           0.91      1600
   macro avg       0.92      0.91      0.91      1600
weighted avg       0.92      0.91      0.91      1600


Confusion matrix:
[[319  17  61   3]
 [  8 342  40  10]
 [  0   0 400   0]
 [  1   1   1 397]]


## 6. Save the results of testing

In [6]:
with open('test_results.txt', 'w') as f:
    f.write(f"Test accuracy: {test_acc:.4f}\n")
    f.write(f"Test loss: {test_loss:.4f}\n")
    f.write("\nClassification Report:\n")
    f.write(classification_report(y_true, y_pred, target_names=class_names))
    f.write("\nConfusion matrix:\n")
    f.write(str(confusion_matrix(y_true, y_pred)))
print("Results saved to test_results.txt")

Results saved to test_results.txt
